# CSC790 Capstone, Week 1: Data, Code-Mixing Scores, Annotation Sheets, Baseline

**Project:** Deep Neural Networks for Semantic Sentiment Analysis in Code-Mixed English-Yoruba Social Media Conversations

This notebook does four things:

1. Loads the AfriSenti/NaijaSenti Yoruba tweets (positive, neutral, negative).
2. Scores every tweet with a rough **Code-Mixing Index (CMI)**.
3. Builds the two **annotation sheets** (one per team member, with 150 shared rows for Cohen's kappa).
4. Fine-tunes the **AfriBERTa-large baseline** and reports results by code-mixing level.

Run on Google Colab or Kaggle with a GPU (Runtime > Change runtime type > T4 GPU). Parts 1 to 3 run without a GPU.

> The CMI here is a lexicon heuristic, good enough for sampling. Later we replace it with a trained word-level language tagger (YECS corpus).

## 0. Setup

In [ ]:
!pip -q install english-words transformers datasets accelerate evaluate scikit-learn openpyxl

In [ ]:
import re, csv, unicodedata, random
import numpy as np, pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED)

BASE = "https://raw.githubusercontent.com/afrisenti-semeval/afrisent-semeval-2023/main/data/yor"
STOP = "https://raw.githubusercontent.com/hausanlp/NaijaSenti/main/data/stopwords/yor.csv"

## 1. Load the Yoruba data

In [ ]:
def load(split):
    df = pd.read_csv(f"{BASE}/{split}.tsv", sep="\t", quoting=csv.QUOTE_NONE, on_bad_lines="skip")
    df["split"] = split
    return df

df = pd.concat([load(s) for s in ["train", "dev", "test"]], ignore_index=True)
df["tweet"] = df["tweet"].astype(str).str.replace('"', '', regex=False).str.strip()
df["id"] = [f"yor_{i:05d}" for i in range(len(df))]

print(df.groupby("split").size())
print(pd.crosstab(df.split, df.label, normalize="index").round(3))

In [ ]:
# Quick look: length and how many tweets carry Yoruba diacritics
df["n_words"] = df.tweet.str.split().str.len()
df["has_diacritics"] = df.tweet.str.contains(r"[ẹọṣàáèéìíòóùúńǹẸỌṢ]")
print(df.n_words.describe().round(1))
print("Share with Yoruba diacritics:", round(df.has_diacritics.mean(), 3))
df.sample(5, random_state=SEED)[["tweet", "label"]]

## 2. Rough Code-Mixing Index (CMI)

Each token is tagged **en** (English word list), **yo** (Yoruba diacritics, Yoruba stopword, or unknown word), or **univ** (mentions, hashtags, numbers, emoji, punctuation).

CMI (Das and Gambäck, 2014) = 100 × (1 − max(en, yo) / (n − univ)). A score of 0 means monolingual; higher means more mixed.

In [ ]:
from english_words import get_english_words_set

def strip_marks(w):
    return "".join(c for c in unicodedata.normalize("NFD", w) if unicodedata.category(c) != "Mn")

ENG = {w for w in get_english_words_set(["web2"], lower=True, alpha=True) if len(w) >= 3}
ENG |= {"lol", "omg", "pls", "abeg", "sha", "wey", "dey", "abi", "guy", "guys", "bro", "okay", "ok", "thanks", "u", "ur", "im", "dont", "cant", "wont"}

yo_stop = pd.read_csv(STOP)["word"].astype(str).str.lower().tolist()
YOR = set(yo_stop) | {strip_marks(w) for w in yo_stop}
# Common Yoruba words that collide with English dictionary entries
YOR |= {"ni", "si", "o", "e", "a", "mi", "wa", "fun", "ko", "lo", "ba", "ti", "to", "se", "je", "re", "won", "omo", "oba", "owo", "ile", "ise", "oro", "ogun", "ojo", "awo", "ewe", "obi", "ore", "ara"}
YO_MARKS = re.compile(r"[ẹọṣàáèéìíòóùúńǹ]")

def tag_tokens(text):
    tags = []
    for tok in text.lower().split():
        if tok.startswith(("@", "#", "http")) or tok == "rt":
            tags.append("univ"); continue
        w = re.sub(r"[^\w']", "", tok)
        if not w or w.isdigit():
            tags.append("univ"); continue
        if YO_MARKS.search(w) or w in YOR:
            tags.append("yo")
        elif w in ENG:
            tags.append("en")
        else:
            tags.append("yo")   # unknown ASCII words in this corpus are mostly unaccented Yoruba
    return tags

def cmi(text):
    t = tag_tokens(text)
    n = len(t); u = t.count("univ")
    if n - u == 0:
        return 0.0, 0, 0
    en, yo = t.count("en"), t.count("yo")
    return round(100 * (1 - max(en, yo) / (n - u)), 1), en, yo

out = df.tweet.apply(cmi)
df["cmi"] = [o[0] for o in out]; df["n_en"] = [o[1] for o in out]; df["n_yo"] = [o[2] for o in out]
df["cmi_bucket"] = pd.cut(df.cmi, [-0.1, 0, 20, 35, 100], labels=["0 (mono)", "0-20", "20-35", "35+"])

print(df.cmi_bucket.value_counts().sort_index())
print("\nLikely code-mixed (CMI > 20 and at least 2 English words):",
      ((df.cmi > 20) & (df.n_en >= 2)).sum())

In [ ]:
# Eyeball a few high-CMI tweets: are they genuinely mixed?
for t in df[df.cmi > 30].sample(8, random_state=1).tweet:
    print("-", t[:180])

## 3. Build the annotation sheets

We sample **1,200 tweets**: 900 that look mixed (spread over CMI buckets) and 300 with CMI 0 as controls.

- **150 shared rows** go to both annotators (for Cohen's kappa).
- The remaining 1,050 are split, so each person labels **675 tweets**.
- Columns to fill: `is_codemixed` (1 or 0), `dominance` (YO, EN, BAL), `notes`.

**Labelling rules (agree on these before you start):**
- **1 (mixed):** the writer naturally switches between Yoruba and English within or across sentences, e.g. *"Olúwa please, better make pizza at home, a ò ní dààmú"*.
- **0 (not mixed):** Yoruba followed by its English **translation** (proverb accounts, "word of the day"); English only in hashtags, links or names; single loanwords that are normal in Yoruba speech (e.g. "phone", "okay"); or fully monolingual text.
- Write `TRANS` in `notes` for translation pairs, so we can report them separately.
- `dominance`: YO if most content words are Yoruba, EN if most are English, BAL if roughly equal. Leave blank when `is_codemixed` is 0.
- Judge the text only; ignore the sentiment. When unsure, write `?` in notes and keep going.

Sampled tweets come from all three splits, and the split column is kept, so the verified test-set rows become your **code-mixed evaluation subset**.

In [ ]:
pool_mixed = df[df.cmi > 0]
pool_mono  = df[df.cmi == 0]

mixed = (pool_mixed.groupby("cmi_bucket", group_keys=False, observed=True)
         .apply(lambda g: g.sample(min(len(g), 300), random_state=SEED)))
mixed = mixed.sample(min(900, len(mixed)), random_state=SEED)
mono = pool_mono.sample(300, random_state=SEED)

sample = pd.concat([mixed, mono]).sample(frac=1, random_state=SEED).reset_index(drop=True)
cols = ["id", "split", "tweet", "label", "cmi"]
sample = sample[cols]
for c in ["is_codemixed", "dominance", "notes"]:
    sample[c] = ""

shared = sample.iloc[:150].copy(); shared["shared"] = 1
rest = sample.iloc[150:].copy(); rest["shared"] = 0
half = len(rest) // 2
sheet_a = pd.concat([shared, rest.iloc[:half]]).sample(frac=1, random_state=1)
sheet_b = pd.concat([shared, rest.iloc[half:]]).sample(frac=1, random_state=2)

# Hide the CMI and sentiment label from annotators so they judge the text alone
hide = ["cmi", "label"]
sheet_a.drop(columns=hide).to_excel("annotation_A.xlsx", index=False)
sheet_b.drop(columns=hide).to_excel("annotation_B.xlsx", index=False)
sample.to_csv("annotation_master.csv", index=False)   # keep this one; it holds labels and CMI
print(len(sheet_a), len(sheet_b), "rows; shared:", int(shared.shared.sum()))

### After annotation: agreement and the code-mixed test subset
Upload both completed sheets, then run this cell.

In [ ]:
from sklearn.metrics import cohen_kappa_score

a = pd.read_excel("annotation_A.xlsx"); b = pd.read_excel("annotation_B.xlsx")

if a.is_codemixed.isna().all() or b.is_codemixed.isna().all():
    print("The annotation sheets are still empty. That's fine: skip this cell for now and come back after you both finish labelling.")
else:
    s = a[a.shared == 1].merge(b[b.shared == 1], on="id", suffixes=("_a", "_b"))
    missing = s.is_codemixed_a.isna() | s.is_codemixed_b.isna()
    if missing.any():
        print(missing.sum(), "shared rows are still blank in one of the sheets; they are left out of the score.")
    s = s[~missing]
    print("Agreement score (Cohen's kappa):", round(cohen_kappa_score(s.is_codemixed_a.astype(int), s.is_codemixed_b.astype(int)), 3))
    print("Rows where you disagree (discuss and settle these):", (s.is_codemixed_a != s.is_codemixed_b).sum())

    # Combine into one verified file (for shared rows, annotator A's answer is kept after you settle disagreements)
    verified = pd.concat([a[a.shared == 0], b[b.shared == 0],
                          s.rename(columns={"is_codemixed_a": "is_codemixed", "dominance_a": "dominance"})])
    verified = verified[["id", "is_codemixed", "dominance"]].dropna(subset=["is_codemixed"])
    verified = verified.merge(pd.read_csv("annotation_master.csv")[["id", "split", "label", "cmi"]], on="id")
    verified.to_csv("verified_codemixed.csv", index=False)
    print(verified.groupby(["split", "is_codemixed"]).size())

## 4. Baseline: AfriBERTa-large

Settings follow the NaijaSenti paper (batch 32, max length 128, learning rate 5e-5); we use fewer epochs with early stopping to save GPU time. Target to reproduce: about **80 weighted F1** on Yoruba.

Run with 3 seeds (42, 43, 44) for the final report.

In [ ]:
import torch, evaluate
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback, set_seed)
from sklearn.metrics import f1_score, classification_report

MODEL = "castorini/afriberta_large"
LABELS = ["negative", "neutral", "positive"]
l2i = {l: i for i, l in enumerate(LABELS)}

tok = AutoTokenizer.from_pretrained(MODEL)
tok.model_max_length = 512

def to_ds(split):
    d = df[df.split == split][["id", "tweet", "label"]].copy()
    d["labels"] = d.label.map(l2i)
    ds = Dataset.from_pandas(d.reset_index(drop=True))
    return ds.map(lambda b: tok(b["tweet"], truncation=True, max_length=128), batched=True,
                  remove_columns=["id", "tweet", "label"])

train_ds, dev_ds, test_ds = to_ds("train"), to_ds("dev"), to_ds("test")
print(torch.cuda.is_available(), len(train_ds), len(dev_ds), len(test_ds))

In [ ]:
def metrics(p):
    preds = p.predictions.argmax(-1)
    return {"weighted_f1": f1_score(p.label_ids, preds, average="weighted"),
            "macro_f1": f1_score(p.label_ids, preds, average="macro")}

def run(seed):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3)
    args = TrainingArguments(
        output_dir=f"out_afriberta_{seed}", per_device_train_batch_size=32, per_device_eval_batch_size=64,
        learning_rate=5e-5, num_train_epochs=6, eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="weighted_f1", save_total_limit=1,
        fp16=torch.cuda.is_available(), report_to="none", seed=seed)
    tr = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=dev_ds,
                 processing_class=tok, compute_metrics=metrics,
                 callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    tr.train()
    pred = tr.predict(test_ds)
    return pred.predictions.argmax(-1)

preds = run(SEED)
test = df[df.split == "test"].reset_index(drop=True).copy()
test["pred"] = [LABELS[i] for i in preds]
print(classification_report(test.label, test.pred, digits=3))

### Results by code-mixing level (first look at Research Question 1)
Uses the rough CMI now; rerun with `verified_codemixed.csv` once annotation is done.

In [ ]:
def wf1(g):
    return pd.Series({"n": len(g), "weighted_f1": f1_score(g.label, g.pred, average="weighted")})

print(test.groupby("cmi_bucket", observed=True).apply(wf1).round(3))

# Bootstrap 95% CI for the mixed vs mono gap (test subsets are small)
def boot_gap(t, B=1000):
    m, o = t[t.cmi > 20], t[t.cmi == 0]
    gaps = []
    for _ in range(B):
        ms, os_ = m.sample(len(m), replace=True), o.sample(len(o), replace=True)
        gaps.append(f1_score(os_.label, os_.pred, average="weighted") - f1_score(ms.label, ms.pred, average="weighted"))
    return np.percentile(gaps, [2.5, 50, 97.5]).round(3)

print("Mono minus mixed F1 gap [2.5%, median, 97.5%]:", boot_gap(test))
test.to_csv(f"preds_afriberta_seed{SEED}.csv", index=False)

## Next steps
1. Both annotators fill their sheets (about 1.5 to 2 hours each), then run the agreement cell.
2. Rerun section 4 with seeds 43 and 44.
3. Week 2: LSTM and BiLSTM-with-attention baselines, then XLM-R and AfroXLMR.